In [1]:
import math
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
from Common.Utils import save_training_results


In [2]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    # System Parameters
    NUM_VIDEOS = 50  # Reduced from 500 for demo speed
    CACHE_CAPACITY_PERCENT = 0.10  # 10% cache
    CACHE_SIZE = int(NUM_VIDEOS * CACHE_CAPACITY_PERCENT) # C
    TILES_PER_VIEWPORT = 4 # k
    
    # State/Action Dimensions
    # State: 10*C + 2 (See Source 468)
    # Action: 5*C + 1 (See Source 469)
    STATE_DIM = 10 * CACHE_SIZE + 2
    ACTION_DIM = 5 * CACHE_SIZE + 1
    
    # DRL Parameters
    GAMMA = 0.6            # Discount factor (Source 473)
    EPSILON = 0.05         # E-greedy parameter (Source 472)
    LR = 0.001             # Learning rate (Source 472)
    BUFFER_SIZE = 2000     # Experience replay buffer size (Source 473)
    BATCH_SIZE = 32        # Mini-batch size (Source 473)
    TARGET_UPDATE_FREQ = 200 # Update target every 200 steps (Source 473)
    TRAIN_EPOCHS = 100     # (Source 471)
    
    # History Windows for State Features
    H_SHORT = 300
    H_LONG = 1000
    
    # Reward Constants (PSNR in dB)
    R_BASE = 30.0    # Reward for serving Base Layer (Source 392)
    R_ENH = 10.0     # Reward for serving Enhancement Layer (Source 392)
    PENALTY = 0.0    # Cost for fetching from backhaul (implicit in lack of reward)

    n_episodes = 300
    n_nodes = 3
    n_users = 1
    step_size = 10.00
    arrival_rate = 10.0  # users per second
    alpha = 0.5
    n_videos = 500
    n_gops = 30
    n_layers = 2
    n = 4
    m = 3
    n_tiles = n * m
    max_capacity = 500e6  # 500 MB

    # Hyperparameters for RL
    epsilon_start = 1.0
    epsilon_min = 0.005
    epsilon_decay = 0.987
    gamma = 0.99
    learning_rate = 1e-3
    batch_size = 32
    capacity = 10000
    window_len = 3  # LSTM sequence length (history window)

    # CPT parameters
    theta = 0.5
    lam = 3.7183

In [3]:
# --- 2. DEEP Q-NETWORK (Section VI & VII-B) ---
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        # 4 Fully Connected Layers (Input, 2 Hidden, Output)
        # Hidden layers have 5C+1 nodes (Source 469)
        hidden_dim = output_dim 
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        
        return self.fc3(x) # Linear activation for output (Source 470)

In [4]:
# --- 3. REPLAY BUFFER ---
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)
    
# --- 4. DRL AGENT ---
class DRLAgent:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Evaluation and Target Networks
        self.policy_net = DQN(config.STATE_DIM, config.ACTION_DIM).to(self.device)
        self.target_net = DQN(config.STATE_DIM, config.ACTION_DIM).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=config.LR)
        self.memory = ReplayBuffer(config.BUFFER_SIZE)
        self.steps_done = 0
        
    def select_action(self, state):
        # Epsilon-Greedy Policy (Source 313)
        if random.random() < self.config.EPSILON:
            return random.randint(0, self.config.ACTION_DIM - 1)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.policy_net(state_t)
                return q_values.argmax().item()

    def train(self):
        if len(self.memory) < self.config.BATCH_SIZE:
            return None # Not enough samples yet

        # Sample mini-batch (Source 328)
        states, actions, rewards, next_states, dones = self.memory.sample(self.config.BATCH_SIZE)

        state_batch = torch.FloatTensor(np.array(states)).to(self.device)
        action_batch = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        reward_batch = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_state_batch = torch.FloatTensor(np.array(next_states)).to(self.device)
        done_batch = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # Compute Q(s, a)
        curr_q_values = self.policy_net(state_batch).gather(1, action_batch)

        # Compute Max Q(s', a') from Target Net (Fixed Target Mechanism)
        next_q_values = self.target_net(next_state_batch).max(1)[0].unsqueeze(1)
        expected_q_values = reward_batch + (self.config.GAMMA * next_q_values * (1 - done_batch))

        # Loss Function (MSE) (Source 349)
        loss = nn.MSELoss()(curr_q_values, expected_q_values)

        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update Target Network periodically (Source 335)
        self.steps_done += 1
        if self.steps_done % self.config.TARGET_UPDATE_FREQ == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        return loss.item()

In [5]:
class AgentAdapter:
    """
    Wraps the provided LabEnv to make it compatible with the DQN Agent.
    Handles:
    1. Feature Extraction (converting raw Env observations to State Vectors).
    2. Action Decoding (converting DQN integer actions to Env commands).
    3. History Tracking (Short vs Long term memory).
    """
    def __init__(self, lab_env, config):
        self.env = lab_env
        self.c = config
        
        # History Tracking (Required for State Features x, y, z)
        # We need to track these externally if the LabEnv doesn't provide processed features
        self.video_history_short = deque(maxlen=self.c.H_SHORT)
        self.video_history_long = deque(maxlen=self.c.H_LONG)
        self.tile_history_short = deque(maxlen=self.c.H_SHORT)
        self.tile_history_long = deque(maxlen=self.c.H_LONG)

        self.video_short_count = defaultdict(int)
        self.video_long_count  = defaultdict(int)


    def update_history(self, request):
        """
        Updates the history queues based on the new request from LabEnv.
        current_request: Expected to contain {video_id, tile_ids}
        """
        vid_id = request['video']
        gop_id = request['gop']
        tile_ids = request['tiles']

        # ---- SHORT window ----
        if len(self.video_history_short) == self.video_history_short.maxlen:
            old = self.video_history_short.popleft()
            self.video_short_count[old] -= 1
            if self.video_short_count[old] == 0:
                del self.video_short_count[old]

        self.video_history_short.append(vid_id)
        self.video_short_count[vid_id] += 1

        # ---- LONG window ----
        if len(self.video_history_long) == self.video_history_long.maxlen:
            old = self.video_history_long.popleft()
            self.video_long_count[old] -= 1
            if self.video_long_count[old] == 0:
                del self.video_long_count[old]

        self.video_history_long.append(vid_id)
        self.video_long_count[vid_id] += 1
        
    def get_state_vector(self, candidate_id, is_tile_candidate, tile_id=None):
        """
        Constructs the neural network input vector (10C + 2) from the LabCacheEngine status
        and our local history queues.
        """
        state = []

        # Access the current cache state from your Engine
        # Assuming env.cache_engine.get_slots() returns list of cached Video IDs
        current_cache_slots = self.env.mec_cache.get_slots() 
        current_cache_tiles = self.env.mec_cache.get_cached_tiles() # Should return list of lists

        print("Current Cache Slots:", current_cache_slots)
        print("Current Cache Tiles:", current_cache_tiles)

        return np.array(state, dtype=np.float32)

    def reset(self):
        _, info = self.env.reset()
        
        # Clear history queues
        self.video_history_short.clear()
        self.video_history_long.clear()
        self.tile_history_short.clear()
        self.tile_history_long.clear()
        
        # Initial State Vector
        state = self.get_state_vector(candidate_id=None, is_tile_candidate=False)
        
        return state, info

In [6]:
def build_state(
    adapter,
    env_cache,
    user
):
    """
    Builds S = (X_s, X_l, Y_s, Y_l, Z_s, Z_l)
    """
    current_video_id = user['video']
    
    C = env_cache.unit_mapper.max_units
    K = env_cache.unit_mapper.viewport_tiles

    # ---- Order cached videos (LRU order) ----
    cached_videos = env_cache.access_order[::-1]  # Most recently used first
    Xs, Xl = [], []
    Ys, Yl = [], []

    for video_id, meta in cached_videos[:C]:
        # ---- X features ----
        Xs.append(adapter.video_short_count.get(video_id, 0))
        Xl.append(adapter.video_long_count.get(video_id, 0))

        # ---- Y features (virtual viewport only) ----
        viewport_tiles = meta["viewport_tiles"]

        for tile_id in viewport_tiles[:K]:
            Ys.append(adapter.tile_short_count.get(tile_id, 0))
            Yl.append(adapter.tile_long_count.get(tile_id, 0))

    # ---- Padding (important) ----
    while len(Xs) < C:
        Xs.append(0)
        Xl.append(0)

    while len(Ys) < C * K:
        Ys.append(0)
        Yl.append(0)

    # ---- Z features (candidate) ----
    Zs = adapter.video_short_count.get(current_video_id, 0)
    Zl = adapter.video_long_count.get(current_video_id, 0)

    state = np.array(
        Xs + Xl + Ys + Yl + [Zs, Zl],
        dtype=np.float32
    )

    return state

In [8]:
from functools import cache


if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()
    
    # 2. Initialize Environment
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.max_capacity,
        # policy=SvcLruPolicy(max_size=max_capacity)
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.alpha
    )
    
    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )
    
    # 3. Initialize Agent Adapter
    adapter = AgentAdapter(env, cfg)
    
    # 4. Initialize DRL Agent
    agent = DRLAgent(cfg)
    
    ### 5. Training Loop ###
    obs, info = adapter.reset()
    done = False
    episode_reward = 0

    for step in count():
        
        reqs_state = info['users_requests']
        active_users = [
            req  for req in reqs_state if req['gop'] < cfg.n_gops
        ]

        actions = []
        for user in active_users:
            print("Processing User Request:", user)

            video = user['video']
            tiles = user['tiles']
            gop = user['gop']

            adapter.update_history(user)

            state = build_state(
                adapter,
                env.mec_cache,
                user
            )

            action = agent.select_action(state)

            if env.mec_cache.unit_mapper.can_add_unit(env.mec_cache.cached_units):
                env.mec_cache.cache_new_video(video, gop, tiles)
            else:
                # DRL chooses which unit (video) to evict
                action = agent.select_action(state)
                env.mec_cache.evict_video()
                env.mec_cache.cache_new_video(video, gop, tiles)
        
            # --- DECISION 1: CACHE VIDEO (Base Layer) ---
            state_vid = adapter.get_state_vector(
                video, is_tile_candidate=False
            )

            # Check if video is already cached using the Engine
            if not adapter.env.mec_cache.is_video_cached(video, layer=0):
                action_vid = agent.select_action(state_vid)

                # Execute logic in environment
                adapter.decode_and_execute_action(action_vid, video)

                # Observe Reward (Distortion Reduction)
                # You might need to calculate this based on whether it was a hit or miss
                reward_vid = cfg.R_BASE if adapter.env.mec_cache.is_video_cached(video, layer=0) else 0

                # Store Transition
                next_state_vid = adapter.get_state_vector(video, is_tile_candidate=False)
                agent.memory.push(state_vid, action_vid, reward_vid, next_state_vid, False)

                ### Decode action to cache command ###
                if action_vid < cfg.CACHE_SIZE * 5:
                    # Cache the video (Base Layer)
                    adapter.env.mec_cache.cache_video(
                        video, layer=0
                    )

        obs, rewards, done, info = env.step(actions)

        if done:
            break

        print(f"Step {step}, Active Users: {len(active_users)}")
        # print(f"Request State: {reqs_state}")
        # print(f"Action: {actions}")
        # print(f"Next Request State: {reqs_next_state}")
        # print(
        #     f"Reward: {rewards}, "
        #     f"Cache Hits: {enhanced_layer_cache_hits + base_layer_cache_hits}, "
        #     f"Cache Misses: {enhanced_layer_cache_misses + base_layer_cache_misses}"
        # )
        print("-----")
    print("--- Training Completed ---")


--- Starting DRL Caching System ---
Current Cache Slots: [0, 4, 6, 9, 11, 12, 15, 16, 18, 19, 20, 21, 23, 24, 29, 31, 33, 35, 36, 38, 39, 40, 42, 43, 44, 45, 50, 52, 55, 58, 60, 61, 62, 63, 64, 67, 70, 72, 73, 74, 75, 76, 78, 79, 80, 82, 84, 85, 88, 90, 91, 93, 94, 95, 96, 97, 98, 101, 102, 103, 105, 108, 111, 112, 113, 115, 117, 119, 120, 123, 125, 130, 132, 133, 135, 139, 140, 141, 142, 146, 150, 154, 158, 161, 162, 163, 166, 167, 172, 173, 174, 175, 176, 177, 178, 180, 181, 185, 187, 190, 192, 195, 198, 200, 201, 203, 204, 205, 206, 207, 210, 212, 213, 215, 217, 222, 224, 226, 228, 230, 231, 232, 233, 237, 240, 241, 242, 243, 244, 246, 247, 248, 250, 251, 252, 254, 255, 256, 258, 259, 260, 261, 262, 263, 266, 268, 273, 275, 277, 279, 280, 281, 283, 284, 285, 286, 287, 288, 292, 293, 297, 298, 300, 303, 304, 305, 306, 308, 311, 313, 314, 315, 316, 317, 318, 319, 323, 325, 327, 328, 330, 331, 337, 338, 340, 341, 342, 347, 350, 352, 353, 354, 355, 357, 360, 361, 362, 363, 364, 365, 370

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x22 and 52x26)